# Combine and Separate Data from Main Crawler

In [ ]:
# Combine scraped data without detailed location and description to maukerja_compiled.csv
import pandas as pd
import glob
import re
import os
import shutil

os.makedirs("../data/raw_data_compiled", exist_ok=True)

new_files = glob.glob("../data/raw_data/maukerja_*.csv")

if not new_files:
    print("No new files in raw_data.")
else:
    # Load new files
    df_new = pd.concat([pd.read_csv(f) for f in new_files], ignore_index=True)
    print(f"New records loaded: {len(df_new):,}")

    # Load existing compiled if it exists
    compiled_path = "../data/maukerja_compiled.csv"
    if os.path.exists(compiled_path):
        df_existing = pd.read_csv(compiled_path, dtype={'job_id': str})
        print(f"Existing compiled records: {len(df_existing):,}")
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_combined = df_new

    print(f"Total before dedup: {len(df_combined):,}")

    # Extract and normalize job_id
    def extract_job_id(row):
        job_id = str(row['job_id']).strip() if pd.notna(row['job_id']) else ''
        url = str(row['url']).strip() if pd.notna(row['url']) else ''
        if job_id in ('', 'nan') and url not in ('', 'nan'):
            match = re.search(r'jobId=(\d+)', url)
            return match.group(1) if match else 'N/A'
        return job_id if job_id not in ('', 'nan') else 'N/A'

    df_combined['job_id'] = df_combined.apply(extract_job_id, axis=1)
    df_combined['job_id'] = (
        df_combined['job_id']
        .astype(str)
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
        .str.replace(r'\s+', '', regex=True)
    )

    # Dedup
    df_final = df_combined[df_combined['job_id'] != 'N/A'].drop_duplicates(subset=['job_id'], keep='first')
    print(f"Total unique jobs: {len(df_final):,}")

    # Save
    df_final.to_csv(compiled_path, index=False, encoding='utf-8-sig')
    print(f"Saved to {compiled_path}")

    # Move processed files
    for f in new_files:
        filename = os.path.basename(f)
        dest = os.path.join("../data/raw_data_compiled", filename)
        shutil.move(f, dest)
        print(f"  Moved {filename}")

In [ ]:
# Separate files for task delegation
# Futher separate to run on different notebooks at the same time
import pandas as pd

# Config
input_file = "../data/maukerja_compiled.csv"
output_file_prefix = "../data/compiled_no_desc"

df = pd.read_csv(input_file)

# Force string dtype
for col in ["scraped_location", "scraped_description"]:
    if col not in df.columns:
        df[col] = pd.array([""] * len(df), dtype="string")
    df[col] = df[col].astype("string").fillna("")

# Split into has description and no description
has_desc = df[
    (df["scraped_description"] != "") &
    # (df["scraped_description"] != "Job Closed") & # Uncomment to recheck Job Closed
    (df["scraped_description"] != "Error")
].copy()

no_desc = df[
    (df["scraped_description"] == "") |
    # (df["scraped_description"] == "Job Closed") | # Uncomment to recheck Job Closed
    (df["scraped_description"] == "Error")
].copy()

print(f"Has description: {len(has_desc):,}")
print(f"No description:  {len(no_desc):,}")

# Save the has_description file (no need to split, already done)
has_desc.to_csv("../data/compiled_has_desc.csv", index=False, encoding="utf-8-sig")
print(f"Saved compiled_has_desc.csv")

# Split no_desc into 3 equal chunks
chunk_size = len(no_desc) // 3
chunks = [
    no_desc.iloc[:chunk_size],
    no_desc.iloc[chunk_size:chunk_size*2],
    no_desc.iloc[chunk_size*2:]
]

for i, chunk in enumerate(chunks):
    path = f"{output_file_prefix}{i}.csv"
    chunk.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"File {i}: {len(chunk):,} rows → {path}")

# Combine Data After Description Scraping

In [ ]:
# merge rescraped files
import pandas as pd
import glob

scraped_files = glob.glob("../data/compiled_*.csv")
df_scraped = pd.concat([pd.read_csv(f) for f in scraped_files], ignore_index=True)
df_scraped.drop_duplicates(subset=['job_id'], keep='first')
print(len(df_scraped))
df_scraped.to_csv("../data/maukerja_compiled.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Save Records with Description only
input_file = "../data/maukerja_compiled.csv"
output_file = "../data/maukerja_compiled_with_description.csv"

df = pd.read_csv(input_file)

# Force string dtype
for col in ["scraped_location", "scraped_description"]:
    if col not in df.columns:
        df[col] = pd.array([""] * len(df), dtype="string")
    df[col] = df[col].astype("string").fillna("")

# Split into has description and no description
has_desc = df[
    (df["scraped_description"] != "") &
    (df["scraped_description"] != "Job Closed") &
    (df["scraped_description"] != "Error")
].copy()

print(f"Has description: {len(has_desc):,}")

has_desc.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"Saved {output_file}")

# File Converters

In [ ]:
# CSV TO GZIP
import pandas as pd 
filename = "../data/maukerja_compiled_with_description"       # Change file names accordingly <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
df = pd.read_csv(f"{filename}.csv")
df.to_csv(f"{filename}.gz", index=False, compression="gzip")

In [ ]:
# GZIP to CSV
import pandas as pd
filename = "../data/maukerja_compiled_with_description"     # Change file names accordingly <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
df = pd.read_csv(f"{filename}.gz", compression="gzip")
df.to_csv(f"{filename}.csv", index=False)